Imports

In [1]:
import numpy as np
from scipy.linalg import expm
from scipy.sparse import bmat, block_diag, csc_matrix
from scipy.sparse.linalg import spsolve
from scipy.linalg import orthogonal_procrustes


Simulation (general multivariate OU)

In [2]:
def simulate_ou_dfa(
    n_subjects=10,
    T_list=None,
    p=50,
    k=3,
    A_scale=0.1,
    Q_scale=1.0,
    Psi_scale=0.5,
    random_state=42
):
    np.random.seed(random_state)
    if T_list is None:
        T_list = [10] * n_subjects

    # general OU parameters
    A = A_scale * np.eye(k)
    Q = Q_scale * np.eye(k)

    Lambda_true = np.random.randn(p, k)

    Ys, Zs, dts_all = [], [], []

    for T in T_list:
        dt = np.random.uniform(0.5, 1.5, size=T-1)
        dts_all.append(dt)

        z = np.zeros((T, k))
        z[0] = np.random.randn(k)

        for t in range(1, T):
            F = expm(-A * dt[t-1])
            Qd = Q * dt[t-1]   # acceptable for simulation
            z[t] = F @ z[t-1] + np.random.multivariate_normal(np.zeros(k), Qd)

        noise = np.random.multivariate_normal(
            np.zeros(p), Psi_scale * np.eye(p), size=T
        )
        y = z @ Lambda_true.T + noise

        Ys.append(y)
        Zs.append(z)

    return Ys, Zs, Lambda_true, dts_all, A, Q


Build OU precision (block tridiagonal)

In [3]:
def build_ou_precision(A, Q, dt):
    """
    Build block-tridiagonal precision matrix for OU latent process
    """
    k = A.shape[0]
    T = len(dt) + 1

    F = [expm(-A * dti) for dti in dt]
    Qd = [Q * dti for dti in dt]  # exact version can be swapped in

    blocks = [[None] * T for _ in range(T)]

    # first block
    blocks[0][0] = np.linalg.inv(Qd[0])

    for t in range(1, T):
        Qt_inv = np.linalg.inv(Qd[t-1])
        Ft = F[t-1]

        blocks[t][t] = Qt_inv
        blocks[t-1][t-1] += Ft.T @ Qt_inv @ Ft
        blocks[t-1][t] = -Ft.T @ Qt_inv
        blocks[t][t-1] = -Qt_inv @ Ft

    return bmat(blocks, format="csc")


E-step: posterior via sparse precision

In [4]:
def ou_posterior_sparse(Y, Lambda, Psi, A, Q, dt):
    T, p = Y.shape
    k = Lambda.shape[1]

    Psi_inv = np.linalg.inv(Psi)

    # OU prior precision
    J_ou = build_ou_precision(A, Q, dt)

    # observation contribution
    J_obs_blocks = []
    h_blocks = []

    for t in range(T):
        J_obs_blocks.append(Lambda.T @ Psi_inv @ Lambda)
        h_blocks.append(Lambda.T @ Psi_inv @ Y[t])

    J_obs = block_diag(J_obs_blocks, format="csc")
    J = J_ou + J_obs
    h = np.concatenate(h_blocks)

    mu = spsolve(J, h)
    Ez = mu.reshape(T, k)

    # marginal covariances (small T only; scalable version uses sparse inverse diag)
    Sigma = np.linalg.inv(J.toarray())
    Ezz = []

    for t in range(T):
        idx = slice(t*k, (t+1)*k)
        Ezz.append(Sigma[idx, idx] + np.outer(Ez[t], Ez[t]))

    return Ez, Ezz


M-step (unchanged from your code)

In [5]:
def update_Lambda(Ez_all, Ezz_all, Ys, tau=1.0, lam=1.0, eps=1e-6):
    p = Ys[0].shape[1]
    k = Ez_all[0].shape[1]

    tau = np.full(k, tau)
    lam = np.full(k, lam)

    Ezz_sum = np.zeros((k, k))
    Ezy_sum = np.zeros((k, p))

    for Y, Ez, Ezz in zip(Ys, Ez_all, Ezz_all):
        for t in range(Y.shape[0]):
            Ezz_sum += Ezz[t]
            Ezy_sum += np.outer(Ez[t], Y[t])

    ridge = np.diag(1.0 / (tau * lam**2 + eps))

    Lambda = np.zeros((p, k))
    for j in range(p):
        Lambda[j] = np.linalg.solve(Ezz_sum + ridge, Ezy_sum[:, j])

    return Lambda


def update_Psi(Ys, Ez_all, Lambda, eps=1e-6):
    p = Ys[0].shape[1]
    num = np.zeros(p)
    den = 0

    for Y, Ez in zip(Ys, Ez_all):
        for t in range(Y.shape[0]):
            res = Y[t] - Lambda @ Ez[t]
            num += res**2
            den += 1

    return np.diag(num / den + eps)


EM loop (precision-based)

In [6]:
def run_em_sparse(Ys, dts_all, k=3, n_iter=30):
    p = Ys[0].shape[1]

    Lambda = np.random.randn(p, k)
    Psi = np.eye(p)

    A = 0.1 * np.eye(k)
    Q = np.eye(k)

    for it in range(n_iter):
        Ezs, Ezzs = [], []

        for Y, dt in zip(Ys, dts_all):
            Ez, Ezz = ou_posterior_sparse(Y, Lambda, Psi, A, Q, dt)
            Ezs.append(Ez)
            Ezzs.append(Ezz)

        Lambda = update_Lambda(Ezs, Ezzs, Ys)
        Psi = update_Psi(Ys, Ezs, Lambda)

        print(f"Iter {it:02d} | mean|Λ| = {np.mean(np.abs(Lambda)):.3f}")

    return Lambda


Factor recovery (unchanged)

In [7]:
def factor_recovery(L_true, L_est):
    R, _ = orthogonal_procrustes(L_est, L_true)
    L_aligned = L_est @ R
    return np.corrcoef(L_true.ravel(), L_aligned.ravel())[0, 1]


✅ Test / main (matches your structure)

In [8]:
def main():
    n_subjects = 10
    T_list = [5,4,6,5,7,4,5,6,5,5]

    Ys, Zs, Lambda_true, dts_all, A, Q = simulate_ou_dfa(
        n_subjects=n_subjects,
        T_list=T_list,
        p=5,
        k=3
    )

    Lambda_est = run_em_sparse(Ys, dts_all, k=3, n_iter=25)

    rec = factor_recovery(Lambda_true, Lambda_est)
    print("\nProcrustes-aligned factor recovery:", rec)


if __name__ == "__main__":
    main()


Iter 00 | mean|Λ| = 1.195
Iter 01 | mean|Λ| = 1.113
Iter 02 | mean|Λ| = 1.078
Iter 03 | mean|Λ| = 1.050
Iter 04 | mean|Λ| = 1.024
Iter 05 | mean|Λ| = 1.003
Iter 06 | mean|Λ| = 0.986
Iter 07 | mean|Λ| = 0.970
Iter 08 | mean|Λ| = 0.955
Iter 09 | mean|Λ| = 0.942
Iter 10 | mean|Λ| = 0.929
Iter 11 | mean|Λ| = 0.916
Iter 12 | mean|Λ| = 0.904
Iter 13 | mean|Λ| = 0.893
Iter 14 | mean|Λ| = 0.882
Iter 15 | mean|Λ| = 0.872
Iter 16 | mean|Λ| = 0.862
Iter 17 | mean|Λ| = 0.852
Iter 18 | mean|Λ| = 0.843
Iter 19 | mean|Λ| = 0.834
Iter 20 | mean|Λ| = 0.825
Iter 21 | mean|Λ| = 0.817
Iter 22 | mean|Λ| = 0.808
Iter 23 | mean|Λ| = 0.801
Iter 24 | mean|Λ| = 0.793

Procrustes-aligned factor recovery: 0.9817775824793535
